# 🔍 02 - Résolution de CAPTCHAs

Ce notebook présente les différents modèles de résolution.

**Modèles disponibles:**
- CRNN (Hana) - 98% accuracy, 19 chars
- TrOCR - 99% accuracy, full charset
- Florence-2 - Zero-shot VLM
- EasyOCR - OCR généraliste

**M2 MoSEF - Université Paris 1 Panthéon-Sorbonne**

In [ ]:
# Imports
import sys
sys.path.insert(0, '..')

import io
import matplotlib.pyplot as plt
from PIL import Image

from app.services.captcha_generator import CaptchaGenerator
from app.services.solver_service import SolverService

In [ ]:
# Initialiser les services
generator = CaptchaGenerator()
solver = SolverService()

print(f"Modèles disponibles: {solver.available_models}")
print(f"Modèles chargés: {solver.loaded_models}")

## 1. Résolution simple avec TrOCR

In [ ]:
# Générer un CAPTCHA
image, true_text = generator.generate(length=5, noise_level=0.3)

# Convertir en bytes
buffer = io.BytesIO()
image.save(buffer, format="PNG")
image_bytes = buffer.getvalue()

# Afficher
plt.figure(figsize=(8, 3))
plt.imshow(image)
plt.title(f"CAPTCHA à résoudre: {true_text}")
plt.axis('off')
plt.show()

In [ ]:
# Résoudre avec TrOCR
result = solver.solve(image_bytes, model="trocr")

print(f"Prédiction: {result.text}")
print(f"Confiance: {result.confidence:.1%}" if result.confidence else "Confiance: N/A")
print(f"Temps: {result.processing_time_ms:.0f} ms")
print(f"Correct: {result.text.lower() == true_text.lower()}")

## 2. Résolution en cascade

In [ ]:
# Nouveau CAPTCHA
image, true_text = generator.generate(length=5, noise_level=0.4)
buffer = io.BytesIO()
image.save(buffer, format="PNG")
image_bytes = buffer.getvalue()

# Résolution en cascade
result = solver.solve_cascade(image_bytes)

plt.figure(figsize=(8, 3))
plt.imshow(image)
plt.title(f"Réel: {true_text} | Prédit: {result.text}")
plt.axis('off')
plt.show()

print(f"\nModèle utilisé: {result.model}")
print(f"Confiance: {result.confidence:.1%}" if result.confidence else "Confiance: N/A")
print(f"Modèles essayés: {result.metadata.get('cascade_tried', [])}")

## 3. Comparaison des modèles

In [ ]:
# Comparer tous les modèles
image, true_text = generator.generate(length=5, noise_level=0.3)
buffer = io.BytesIO()
image.save(buffer, format="PNG")
image_bytes = buffer.getvalue()

compare_result = solver.compare(image_bytes, true_text=true_text)

plt.figure(figsize=(8, 3))
plt.imshow(image)
plt.title(f"CAPTCHA: {true_text}")
plt.axis('off')
plt.show()

print("\n📊 Résultats de la comparaison:\n")
for name, res in compare_result.results.items():
    is_correct = res.text.lower() == true_text.lower()
    status = "✅" if is_correct else "❌"
    conf = f"{res.confidence:.1%}" if res.confidence else "N/A"
    print(f"{status} {name.upper():10} | {res.text:10} | Conf: {conf:6} | {res.processing_time_ms:.0f}ms")

print(f"\n🏆 Gagnant: {compare_result.winner}")

## 4. Test sur plusieurs CAPTCHAs

In [ ]:
# Tester sur 5 CAPTCHAs
n_tests = 5
results = []

fig, axes = plt.subplots(1, n_tests, figsize=(15, 3))

for i, ax in enumerate(axes):
    image, true_text = generator.generate(length=5, noise_level=0.3)
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    image_bytes = buffer.getvalue()
    
    result = solver.solve(image_bytes, model="trocr")
    is_correct = result.text.lower() == true_text.lower()
    results.append(is_correct)
    
    ax.imshow(image)
    color = "green" if is_correct else "red"
    ax.set_title(f"R:{true_text} P:{result.text}", color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

accuracy = sum(results) / len(results) * 100
print(f"\nAccuracy sur {n_tests} tests: {accuracy:.0f}%")

## 5. Informations sur les modèles

In [ ]:
# Informations détaillées
model_info = solver.get_model_info()

for name, info in model_info.items():
    print(f"\n🤖 {name.upper()}")
    print(f"   Type: {info['type']}")
    print(f"   Charset: {info['charset_size']} caractères")
    print(f"   Chargé: {'✅' if info['is_loaded'] else '⬜'}")
    print(f"   Disponible: {'✅' if info['is_available'] else '❌'}")

---
**Fin du notebook 02**